# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

Prioritize pages that have enough search visibility but weak CTR while still ranking within a reasonably visible search position. Give extra priority to pages that are also stale.

The rule uses four signals:
- `impressions_90d >= 731` → enough search visibility
- `ctr <= 0.07` → weak CTR
- `avg_position <= 22.3` → reasonably visible search position
- `days_since_last_update >= 104` → stale

### Reason codes

- `weak_ctr_visible` → the page has meaningful impressions but weak CTR.
- `weak_ctr_good_position` → the page has weak CTR despite still ranking within a reasonably visible position.
- `stale_weak_ctr` → the page has weak CTR and has not been updated recently.
- `stale_weak_ctr_visible` → the page has enough visibility, weak CTR, and is stale, making it a higher-priority review candidate.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
print("=== BASELINE SIGNALS ===")

print("\nImpressions (90d):")
print(df["impressions_90d"].describe())

print("\nCTR:")
print(df["ctr"].describe())

print("\nAverage Position:")
print(df["avg_position"].describe())

print("\nDays Since Last Update:")
print(df["days_since_last_update"].describe())

=== BASELINE SIGNALS ===

Impressions (90d):
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

CTR:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

Average Position:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

Days Since Last Update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64


In [4]:
print(df["avg_position"].describe())

count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64


In [5]:
# Build the baseline score

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

# Weak CTR relative to search position
weak_ctr = (
    (df["ctr"] < 0.5) &
    (df["avg_position"] <= 20)
).astype(int)

# Score: visibility + weak performance + staleness
df["score"] = visible * weak_ctr * (1 + stale)

# Reason codes
df["reason_code"] = "not_flagged"

df.loc[
    (visible == 1) & (weak_ctr == 1),
    "reason_code"
] = "weak_ctr_visible"

df.loc[
    (visible == 1) & (weak_ctr == 1) & (stale == 1),
    "reason_code"
] = "weak_ctr_visible_stale"

# Rank highest-priority pages first
ranked = df.sort_values("score", ascending=False)

print(ranked[[
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "score",
    "reason_code"
]].head(20))

                 content_id  impressions_90d   ctr  avg_position  \
26799  content_77d4d5930e5e              828  0.24          18.6   
5327   content_fe16a55cd13d             4556  0.33          16.4   
11630  content_6226ee6adc91              545  0.18          17.8   
22872  content_e3ff1b093148             1408  0.28           7.8   
26840  content_7f116ae1f6f5              954  0.42           9.0   
16751  content_cf56e2e2e282            61678  0.15          19.7   
7452   content_72496874f806              821  0.24           5.8   
12045  content_c2d929d83eaa             7558  0.20          17.9   
20837  content_928af3e22c80             1697  0.12          15.8   
21268  content_0a91db491d14            13299  0.49          10.5   
10293  content_68b7e392aee5              793  0.00          12.9   
10189  content_5d69d55ace24             3541  0.42           5.2   
10169  content_99cca44e45a6             5067  0.32           6.3   
23142  content_7c23a548965d            14004  0.

In [6]:
import os

os.makedirs("../outputs", exist_ok=True)

df.to_csv("../outputs/baseline_action_score.csv", index=False)

print("CSV saved successfully.")

CSV saved successfully.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = df.sort_values("score", ascending=False).head(20)

print(top20[
    [
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "score",
        "reason_code"
    ]
].to_string(index=False))

          content_id  impressions_90d  ctr  avg_position  days_since_last_update  score            reason_code
content_77d4d5930e5e              828 0.24          18.6                     194      2 weak_ctr_visible_stale
content_fe16a55cd13d             4556 0.33          16.4                     194      2 weak_ctr_visible_stale
content_6226ee6adc91              545 0.18          17.8                     183      2 weak_ctr_visible_stale
content_e3ff1b093148             1408 0.28           7.8                     183      2 weak_ctr_visible_stale
content_7f116ae1f6f5              954 0.42           9.0                     301      2 weak_ctr_visible_stale
content_cf56e2e2e282            61678 0.15          19.7                     194      2 weak_ctr_visible_stale
content_72496874f806              821 0.24           5.8                     301      2 weak_ctr_visible_stale
content_c2d929d83eaa             7558 0.20          17.9                     193      2 weak_ctr_visible_stale
c

In [8]:
# Section 3: Top-20 review

review = top20[
    [
        "content_id",
        "reason_code",
        "score",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

review["action"] = "Review page for possible update"

review["confidence_note"] = (
    "Medium confidence: the page meets the baseline conditions, "
    "but additional page/query context is needed."
)

review["what_would_make_it_wrong"] = (
    "CTR may be normal for the specific queries, page type, or SERP context."
)

review = review[
    [
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print(review.to_string(index=False))

          content_id                          action            reason_code                                                                                         confidence_note                                                what_would_make_it_wrong
content_77d4d5930e5e Review page for possible update weak_ctr_visible_stale Medium confidence: the page meets the baseline conditions, but additional page/query context is needed. CTR may be normal for the specific queries, page type, or SERP context.
content_fe16a55cd13d Review page for possible update weak_ctr_visible_stale Medium confidence: the page meets the baseline conditions, but additional page/query context is needed. CTR may be normal for the specific queries, page type, or SERP context.
content_6226ee6adc91 Review page for possible update weak_ctr_visible_stale Medium confidence: the page meets the baseline conditions, but additional page/query context is needed. CTR may be normal for the specific queries, page type, or SERP c

I would use this list as a first check rather than assuming every page is actually bad. For example, content_0a91db491d14 has high impressions and a CTR of 0.49, but it is still ranking around position 10.5. I would check the actual search queries before deciding that the page needs an update.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Section 4: Weak picks + leakage check

print("=== WEAK PICK ===")
print(
    "content_0a91db491d14 is a weak/questionable pick because "
    "its CTR is 0.49 despite having high impressions and an average "
    "position of 10.5. The baseline flags it, but query-level context "
    "could show that the CTR is reasonable."
)

print("\n=== LEAKAGE CHECK ===")
print("Baseline uses only current page-level signals:")
print([
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
])

print("\nNo future-window or label-derived variables were used in the baseline.")

=== WEAK PICK ===
content_0a91db491d14 is a weak/questionable pick because its CTR is 0.49 despite having high impressions and an average position of 10.5. The baseline flags it, but query-level context could show that the CTR is reasonable.

=== LEAKAGE CHECK ===
Baseline uses only current page-level signals:
['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update']

No future-window or label-derived variables were used in the baseline.


I checked the baseline inputs and they are based on page-level signals available in the dataset. I did not use future outcomes or labels to create the score. The weak pick above also shows why the baseline should be treated as decision-support rather than proof that a page needs an update.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.